# 预期ST因子测试

- **股票池** (`cn_stock_prefactors_community`)
  - 按 `total_market_cap` 升序排列取前500只
  - `st_status = 0` (非ST)
  - `suspended = 0` (非停牌)
  - `is_bz50 = 0` (非北交所)
- **因子条件** (`cn_stock_factors_financial_items`)
  - `net_profit_lf < 0` (最近完整财年净利润为负)
  - `net_profit_ly < 0` (上一财年净利润为负)
- **持仓**
  - 等权分配: `position = 1.0 / 当日符合条件股票数`
  - 每日调仓

In [3]:
from bigmodule import M, I
import dai
import pandas as pd


def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


def m5_before_trading_start_bigquant_run(context, data):
    pass


def m5_handle_tick_bigquant_run(context, tick):
    pass


def m5_handle_data_bigquant_run(context, data):
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


def m5_handle_trade_bigquant_run(context, trade):
    pass


def m5_handle_order_bigquant_run(context, order):
    pass


def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========
# 预期ST因子：连续两个财政年度净利润为负（老规则）
# 使用 cn_stock_financial_lf_shift 表获取精确的财年净利润

stock_sql = """
WITH 
-- 第一步：从全市场选出市值最小的500只（非ST、非停牌、非北交所）
small_cap_500 AS (
    SELECT
        date,
        instrument,
        total_market_cap
    FROM cn_stock_prefactors_community
    WHERE
        st_status = 0
        AND suspended = 0
        AND is_bz50 = 0
    QUALIFY
        ROW_NUMBER() OVER (PARTITION BY date ORDER BY total_market_cap ASC) <= 500
),
-- 第二步：在小市值500中筛选连续两个财年净利润为负
filtered AS (
    SELECT
        a.date,
        a.instrument,
        a.total_market_cap,
        b.net_profit_lf,
        b.net_profit_ly
    FROM small_cap_500 a
    JOIN cn_stock_factors_financial_items b USING (date, instrument)
    WHERE b.net_profit_lf < 0 AND b.net_profit_ly < 0
)
SELECT
    date,
    instrument,
    total_market_cap,
    net_profit_lf,
    net_profit_ly,
    -total_market_cap AS score,
    1.0 / c_sum(1) AS position
FROM filtered
ORDER BY date, instrument
"""

print("正在查询数据...")
filtered_df = dai.query(stock_sql, filters={"date": ["2020-01-01", "2026-12-31"]}).df()
print(f"满足预期ST条件的记录数：{len(filtered_df)}")
print(f"每日平均持仓数量：{filtered_df.groupby('date')['instrument'].count().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

正在查询数据...
满足预期ST条件的记录数：2124
每日平均持仓数量：7.2
[2026-04-08 13:17:23] [info     ] bigtrader.v30 开始运行 ..
[2026-04-08 13:17:23] [info     ] read input 'data' ..
[2026-04-08 13:17:23] [info     ] 2021-01-01, 2026-04-07, , equity, instruments=642
[2026-04-08 13:17:23] [info     ] bigtrader module V2.1.0
[2026-04-08 13:17:23] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-04-08 13:17:32] [info     ] backtest done, raw_perf_ds:dai.DataSource("_3f957e74c21042d79e2a2e6a9a6b31db")


[2026-04-08 13:17:34] [info     ] bigtrader.v30 运行完成 [11.143s].


In [4]:
# ========== 导出交易记录CSV ==========
raw_perf = m5.raw_perf.read()
trades_list = [t for txns in raw_perf['transactions'] if txns for t in txns]
trades_df = pd.DataFrame(trades_list)

# 获取所有交易涉及的股票
instruments = trades_df['symbol'].unique().tolist()

# 查询股票名称和行业（取最新数据）
info_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    name
FROM cn_stock_prefactors_community
ORDER BY instrument, date DESC
"""
stock_info = dai.query(info_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
stock_info_dict = stock_info.set_index('instrument')['name'].to_dict()

industry_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    industry_level1_name,
    industry_level2_name
FROM cn_stock_industry_component
ORDER BY instrument, date DESC
"""
industry_info = dai.query(industry_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
industry_dict = industry_info.set_index('instrument')[['industry_level1_name', 'industry_level2_name']].to_dict('index')

output_records = []
holdings = {}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    if amount > 0:
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings:
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        ind = industry_dict.get(instrument, {})
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '股票名': stock_info_dict.get(instrument, ''),
            '行业分类': ind.get('industry_level1_name', ''),
            '二级行业': ind.get('industry_level2_name', ''),
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

output_path = './预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到：{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)

交易记录已保存到：./预期ST_bigquant交易记录.csv
共 1763 条交易记录


,股票代码,股票名,行业分类,二级行业,买入日期,卖出日期,买入价格(前复权),卖出价格(前复权),涨幅
1762,000790,华神科技,医药,中药生产,2026-04-01,2026-04-02,4.18,4.19,0.0024
1761,002715,登云股份,汽车,汽车零部件Ⅱ,2026-04-01,2026-04-02,15.48,14.77,-0.0459
1760,688132,邦彦技术,国防军工,其他军工Ⅱ,2026-03-23,2026-03-24,16.58,15.95,-0.0380
1759,920985,海泰新能,电气设备,电源设备,2026-03-02,2026-03-03,7.90,7.81,-0.0114
1758,002072,凯瑞德,商贸零售,贸易Ⅱ,2026-02-12,2026-02-13,7.69,8.03,0.0442
1757,920030,德众汽车,None,None,2025-12-25,2025-12-26,6.73,6.75,0.0030
1736,600791,京能置业,房地产,房地产开发和运营,2025-11-03,2025-11-04,5.13,5.22,0.0175
1742,600448,华纺股份,纺织服装,纺织制造,2025-11-03,2025-11-04,3.57,3.60,0.0084
1741,600303,曙光股份,汽车,商用车,2025-11-03,2025-11-04,3.83,3.87,0.0104
1740,688296,和达科技,计算机,计算机软件,2025-11-03,2025-11-04,15.09,15.04,-0.0033
